# Step 1: Setup and Authentication

This cell installs necessary libraries, connects to your Google Drive (make sure you're logged in Colab and Drive with the same Google account), and enters your Anthropic API Key. The key needs to be acquired beforehand from https://console.anthropic.com/settings/keys and stored in the Colab secret variable ANTHROPIC_API_KEY.

It needs to be run once at the beginning of the session and when the session is restarted.


In [ ]:
# Step 1: Setup and Authentication
# Run this cell to install necessary libraries, connect to your Google Drive, and enter your Anthropic API Key.

# Install required Python libraries
!pip install anthropic pandas openpyxl pypdf -q

import anthropic
from anthropic import Anthropic
from google.colab import drive, userdata
import pandas as pd
import os
import re
from IPython.display import display, Markdown
import time
import datetime

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# MOUNT GOOGLE DRIVE
# This allows the notebook to access your Google Drive files.
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# CONFIGURE ANTHROPIC API KEY
# Get a key from https://console.anthropic.com/settings/keys
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
try:
    # Use Colab's user data feature for secure key storage
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY').strip()
except userdata.SecretNotFoundError:
    from google.colab import output
    output.enable_custom_widget_manager()
    from getpass import getpass
    ANTHROPIC_API_KEY = getpass('Please enter your Anthropic API Key: ')
    userdata.set('ANTHROPIC_API_KEY', ANTHROPIC_API_KEY)

try:
    # Create the Anthropic client.
    # The SDK itself quietly retries rate-limit (429) and overloaded (529)
    # errors twice with exponential back-off. NumberOfRetries in Step 2 is
    # applied on top of that.
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    print("✅ Anthropic API client configured successfully.")
except Exception as e:
    print(f"❌ Error configuring Anthropic API client: {e}")
    print("   Please ensure you have entered a valid API key.")

# Step 2: Define File Paths and Prompt

This cell sets up the prompt and creates:
- output excel file with a given structure,
- excel file with processed file names,
- log file for calls and errors,
- raw output file with the unprocessed answers returned by the API.

It needs to be run at the beginning of the session and every time you change prompt, data subfolder or output structure.

This is the only part of the code that needs changing when the process is applied on new data.

You need to change the following:
- SubFolder - the subfolder where text data are stored (pdf, txt and md files are supported)
- ProjectName - the name of the project; it is used to build the names of the results, processed, log and raw output files
- PDF_FOLDER_PATH - the main folder that contains subfolders with data and where the output files will be created
- NumberOfRetries - how many times a call is repeated if the API returns a rate limit error
- WaitTime - how many seconds to wait before each retry
- CLAUDE_MODEL - Selected Claude model.
- EFFORT - How much work the model puts into each answer, including how much it thinks: `low`, `medium`, `high` (the API default), `xhigh` or `max`. This is the main quality / cost / speed dial. For extraction against a coding manual `low` or `medium` is usually enough; raise it only if the manual asks the model to infer rather than look up.
- THINKING - `adaptive` lets the model decide when and how much to think (this is the default on current models). `disabled` switches thinking off entirely, which is the cheapest and fastest option for pure look-up fields. On Claude Opus 5 thinking cannot be disabled at `xhigh` or `max` effort.
- MAX_OUTPUT_TOKENS - A ceiling on the length of each answer, which also caps the cost of a runaway call. Thinking tokens count towards this limit, so leave headroom.
- SERVICE_TIER - `auto` for normal processing (uses priority capacity if your organisation has any), or `standard_only`. There is no cheap-but-slow tier on a live call; the Claude equivalent is the separate Message Batches API, which is a different workflow and is not implemented here.
- USE_STRICT_SCHEMA - If True the model is forced to return exactly the fields in MODEL_COLUMNS (Claude Structured Outputs) instead of merely being asked to in the prompt.
- USE_PROMPT_CACHE - If True the coding manual, which is identical for every file in a batch, is marked for caching and read back at a fraction of the normal input price on later calls. Unlike OpenAI, Claude does not cache anything unless asked to, so this is worth leaving on. Caching only kicks in once the manual is long enough: about 512 tokens on Opus 5 / Fable 5.1, 1,024 on Sonnet 5 and 4,096 on Haiku 4.5. The `Cached tokens` column tells you whether it is working.
- REPLICATIONS - How many times each file is sent to the model. Leave at 1 for normal use. Set it to 3 or 5 to measure how stable the coding is across runs, which is a reliability check.
- MODEL_COLUMNS - The fields the model must return. These must match the coding manual in the prompt, and the JSON schema is generated from them automatically.
- SCHEMA_FIELD_TYPES - Fields that are not text. Anything not listed is treated as a string.
- METADATA_COLUMNS - Columns the notebook adds itself (source file, model, run number, token counts). You do not normally need to change these.

**A note on `temperature`:** Claude Opus 5, Sonnet 5 and Fable 5.1 reject any non-default `temperature`, `top_p` or `top_k` with an error, whether or not thinking is on. The equivalent control is EFFORT, and consistency of output comes from USE_STRICT_SCHEMA rather than from a low temperature.

**A note on verbosity:** Claude has no verbosity parameter. The strict JSON schema keeps the answer to the fields requested; if free-text fields run long, say so in the coding manual.
- CLAUDE_SYSTEM_PROMPT - System prompt that sets the model's role and asks for valid JSON
- CLAUDE_PROMPT - Structured prompt with instructions for content analysis (the coding manual)
- EXCEL_COLUMNS - The structure of the output Excel file. It is now built automatically as MODEL_COLUMNS + METADATA_COLUMNS, so you only edit MODEL_COLUMNS.

The following paths are derived automatically from PDF_FOLDER_PATH and ProjectName and normally do not need changing:
- EXCEL_FILE_PATH - The output Excel file: `ProjectName - Claude - results.xlsx`
- PROCESSED_FILES_PATH - The Excel file with the names of the files already processed: `ProjectName - Claude - Processed.xlsx`
- LOG_FILE_PATH - The log file: `ProjectName - log.txt`
- RAW_OUTPUT_PATH - The raw API output file: `ProjectName - raw.txt`

In [ ]:
# Step 2: Define File Paths and Prompt
# Verify the folder and file names below, then run this cell.

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# DEFINE FILE AND FOLDER PATHS
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---

# The subfolder that holds the source files to analyse (pdf, txt, md)
SubFolder = "Papers"

# The name of the project. All output file names are built from it (up to 64 characters).
ProjectName = "LLM Workflow Example - Entrepreneurship Diversity 2nd"

# The folder inside your Google Drive that is root project folder
# Results and logs will be written here. Files for processing should be in subfolders
PROJECT_FOLDER_PATH = '/content/drive/MyDrive/LLMExample'

# The folder inside your Google Drive containing the source files
PDF_FOLDER_PATH = os.path.join(PROJECT_FOLDER_PATH, SubFolder)

# Make sure the folders exists before we try to create files in it
os.makedirs(PROJECT_FOLDER_PATH, exist_ok=True)
os.makedirs(PDF_FOLDER_PATH, exist_ok=True)

# The name of the output Excel file (derived from PDF_FOLDER_PATH and ProjectName)
EXCEL_FILE_PATH = os.path.join(PROJECT_FOLDER_PATH, f'{ProjectName} - Claude - results.xlsx')

# The log file for calls and errors
LOG_FILE_PATH = os.path.join(PROJECT_FOLDER_PATH, f'{ProjectName} - Claude - log.txt')

# The file holding the raw output returned by the API
RAW_OUTPUT_PATH = os.path.join(PROJECT_FOLDER_PATH, f'{ProjectName} - Claude - raw.txt')

# The path for the processed files tracking sheet
PROCESSED_FILES_PATH = os.path.join(PDF_FOLDER_PATH, f'{ProjectName} - Claude - Processed.xlsx')

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# RETRY SETTINGS (used when the API returns a rate limit error)
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
NumberOfRetries = 1   # how many times a rate limited call is retried
WaitTime = 1          # how many seconds to wait before each retry

# ------------------------------------------------------------
# Set the model you want to use here
# ------------------------------------------------------------

# CLAUDE_MODEL = 'claude-haiku-4-5-20251001'         # fastest and cheapest;
# CLAUDE_MODEL = 'claude-sonnet-5-5'                 # balanced speed, cost and quality
CLAUDE_MODEL = 'claude-opus-5-5'
# CLAUDE_MODEL = 'claude-fable-5-1'

# ------------------------------------------------------------
# MODEL CONTROLS
# ------------------------------------------------------------
# NOTE ON TEMPERATURE
# Claude Opus 5, Sonnet 5 and Fable 5.1 reject any non-default temperature,
# top_p or top_k with an error, whether or not thinking is on. The dial that
# replaces temperature is EFFORT below, and repeatable output comes from
# USE_STRICT_SCHEMA rather than from a low temperature.

# How much work the model puts into each answer, including how much it thinks.
# low | medium | high (API default) | xhigh | max
# For extraction against a coding manual, low or medium is usually enough.
EFFORT = "high"

# Whether the model thinks before answering.
# adaptive = the model decides when and how much (default on current models)
# disabled = no thinking at all - cheapest and fastest, fine for look-up fields
# Note: on Claude Opus 5, thinking cannot be disabled at xhigh or max effort.
THINKING = "adaptive"

# Ceiling on the length of one answer, and therefore on the cost of one call.
# Thinking tokens count towards this limit, so leave headroom.
# If answers come back cut off, raise this number.
MAX_OUTPUT_TOKENS = 32000

# auto = normal processing (uses priority capacity if your organisation has any)
# standard_only = never use priority capacity
SERVICE_TIER = "auto"

# True  = force the model to return exactly the fields in MODEL_COLUMNS
#         (Claude Structured Outputs). Strongly recommended.
# False = only ask for JSON in the prompt.
USE_STRICT_SCHEMA = True

# Claude does not cache anything unless asked to. True marks the coding manual
# for caching so later calls in the batch read it back at a fraction of the
# normal input price. It only works once the manual is long enough: about
# 512 tokens on Opus 5 / Fable 5.1, 1,024 on Sonnet 5, 4,096 on Haiku 4.5.
USE_PROMPT_CACHE = True

# How many times each file is sent to the model.
# 1 = normal use. Set to 3 or 5 to measure how stable the coding is across
# runs - the reliability check that replaces temperature = 0. Each replication
# is written as a separate row with its own Run number.
REPLICATIONS = 1


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# SYSTEM PROMPT - SETS THE ROLE AND ASKS FOR VALID JSON
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# With USE_STRICT_SCHEMA the JSON structure is enforced by the API, so this
# text is belt-and-braces. It still matters when the schema is off.
CLAUDE_SYSTEM_PROMPT = """
You are a precise data extraction assistant for academic content analysis.
You must answer with valid JSON only.
Return a single JSON object containing the key "results", whose value is an array
of record objects, i.e. {"results": [ { ... }, { ... } ]}.
Every field requested in the coding manual must be present in every record.
Do not add markdown code fences, comments, or any text outside the JSON.
""".strip()

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# DEFINE THE PROMPT FOR CLAUDE - THE CODING MANUAL (JSON OUTPUT)
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# This text is identical for every file in a batch, which is what makes it
# worth caching (USE_PROMPT_CACHE). The paper text is sent separately.
CLAUDE_PROMPT = """
You are a researcher in Management conducting a literature review. You will receive a text of an academic paper. Please use the following coding manual to code the text.
IE_YEAR_PUBLISHED = The year the paper was published.
IE_JOURNAL = Journal in which the paper was published.
IE_AUTHORS = Authors of the paper.
IE_TITLE = Title of the paper.
IE_DOI = DOI of the paper.
AN_RQ = Did the authors explicitly define research questions, research aims, or research objectives? (1 = Yes, 0 = No).
IE_RQ_VERBATIM = Copy verbatim text from the paper that defines the research questions/aims/objectives. If none, return N/A.
SU_FULL_PROCESS = What were the steps of the research and analysis process from start to finish?
AN_PAPER_TYPE = Type of the paper – empirical, review, conceptual, editorial, other.
SU_PAPER_TYPE_REASON = Explain why you chose the paper type.
AN_METHOD_TYPE = What type of research methods were used (qualitative, quantitative, multiple, mixed, review, other)?
SU_METHOD_TYPE_REASON = Explain why you chose the research method type.
IE_METHOD = List the main research method for each study in the paper, separated by comma. Return N/A if none.
AN_HYPOTHESES_PRESENT = Does the paper test any hypotheses (1 = Yes, 0 = No).
IE_HYPOTHESES = Extract verbatim all hypotheses from the paper. Return N/A if none.
SU_DATA_COLLECTION = How was the data collected?
IE_DATA_COLLECTION_VERBATIM = Copy verbatim text from the paper that describes data collection. If there are multiple relevant sections combine them (separated by “…”) in a single text. Return N/A if none.
IE_SAMPLE_DATATYPE = Extract the size of the sample for each data type. For each type of data return [sample-size] ([data-type]), separated by comma. Return N/A if not explicitly stated.
IE_SAMPLE_COUNTRIES = List all countries that were the source of data sample, separated by comma. Return “not reported” if paper is empirical but did not report countries where the data was collected. Return N/A if paper is not empirical.
AN_THEORETICAL_CONTRIBUTION = Was the theoretical contribution explicitly described? (1 = Yes, 0 = No).
IE_THEORETICAL_CONTRIBUTION_VERBATIM = Copy verbatim text from the paper that describes the theoretical contribution of the study. If there are multiple relevant sections combine them (separated by “…”) in a single text. Return N/A if none.

""".strip()

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# INITIALIZE EXCEL FILE
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# Define the columns for the Excel file
# MODEL_COLUMNS = the fields the model must return. They must match the coding
# manual in the prompt above. The JSON schema used for Structured Outputs is
# generated from this list, so changing the manual and this list is all you need.
MODEL_COLUMNS = [
    "IE_YEAR_PUBLISHED", "IE_JOURNAL", "IE_AUTHORS", "IE_TITLE", "IE_DOI", "AN_RQ", "IE_RQ_VERBATIM",
    "SU_FULL_PROCESS", "AN_PAPER_TYPE", "SU_PAPER_TYPE_REASON", "AN_METHOD_TYPE", "SU_METHOD_TYPE_REASON",
    "IE_METHOD", "AN_HYPOTHESES_PRESENT", "IE_HYPOTHESES", "SU_DATA_COLLECTION", "IE_DATA_COLLECTION_VERBATIM",
    "IE_SAMPLE_DATATYPE", "IE_SAMPLE_COUNTRIES", "AN_THEORETICAL_CONTRIBUTION", "IE_THEORETICAL_CONTRIBUTION_VERBATIM"
]

# Fields that are not text. Anything not listed here is treated as a string.
SCHEMA_FIELD_TYPES = {
    "AN_RQ": "integer",
    "AN_THEORETICAL_CONTRIBUTION": "integer",
}

# Columns the notebook adds itself: where the row came from, and what it cost.
METADATA_COLUMNS = [
    "Source file",
    "SubFolder",
    "Model",
    "Timestamp",
    "Run",
    "Effort",
    "Thinking",
    "Input tokens",
    "Cached tokens",
    "Thinking tokens",
    "Output tokens",
]

# The full structure of the output Excel file
EXCEL_COLUMNS = MODEL_COLUMNS + METADATA_COLUMNS

EXCELPROCESSED_COLUMNS = [
    "Filename"
]

# Create the Excel file with headers if it doesn't already exist
if not os.path.exists(EXCEL_FILE_PATH):
    df_initial = pd.DataFrame(columns=EXCEL_COLUMNS)
    df_initial.to_excel(EXCEL_FILE_PATH, index=False)
    print(f"✅ Created new Excel file at: {EXCEL_FILE_PATH}")
else:
    print(f"✅ Excel file already exists at: {EXCEL_FILE_PATH}. New data will be appended.")

# Create the Excel file with headers if it doesn't already exist
if not os.path.exists(PROCESSED_FILES_PATH):
    df_initial = pd.DataFrame(columns=EXCELPROCESSED_COLUMNS)
    df_initial.to_excel(PROCESSED_FILES_PATH, index=False)
    print(f"✅ Created new Excel file at: {PROCESSED_FILES_PATH}")
else:
    print(f"✅ Excel file already exists at: {PROCESSED_FILES_PATH}. New data will be appended.")

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# INITIALIZE LOG FILE AND RAW OUTPUT FILE
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# Both files are opened in append mode later, so existing content is preserved.
_setup_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

if not os.path.exists(LOG_FILE_PATH):
    with open(LOG_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write(f"[{_setup_timestamp}] Log file created for project '{ProjectName}'.\n")
    print(f"✅ Created new log file at: {LOG_FILE_PATH}")
else:
    print(f"✅ Log file already exists at: {LOG_FILE_PATH}. New entries will be appended.")

if not os.path.exists(RAW_OUTPUT_PATH):
    with open(RAW_OUTPUT_PATH, 'w', encoding='utf-8') as f:
        f.write(f"[{_setup_timestamp}] Raw output file created for project '{ProjectName}'.\n")
    print(f"✅ Created new raw output file at: {RAW_OUTPUT_PATH}")
else:
    print(f"✅ Raw output file already exists at: {RAW_OUTPUT_PATH}. New output will be appended.")

# Record the configuration in the log file
with open(LOG_FILE_PATH, 'a', encoding='utf-8') as f:
    f.write(
        f"[{_setup_timestamp}] CONFIGURATION | Project: {ProjectName} | SubFolder: {SubFolder} | "
        f"Model: {CLAUDE_MODEL} | Effort: {EFFORT} | Thinking: {THINKING} | "
        f"Max output tokens: {MAX_OUTPUT_TOKENS} | Service tier: {SERVICE_TIER} | "
        f"Strict schema: {USE_STRICT_SCHEMA} | Prompt cache: {USE_PROMPT_CACHE} | "
        f"Replications: {REPLICATIONS} | NumberOfRetries: {NumberOfRetries} | WaitTime: {WaitTime}s\n"
    )

# Display configuration for user verification
display(Markdown(f"""
### Configuration Set:
* **Project name:** `{ProjectName}`
* **SubFolder:** `{SubFolder}`
* **Source file folder:** `{PDF_FOLDER_PATH}`
* **Output Excel File:** `{EXCEL_FILE_PATH}`
* **Processed file list:** `{PROCESSED_FILES_PATH}`
* **Log file:** `{LOG_FILE_PATH}`
* **Raw output file:** `{RAW_OUTPUT_PATH}`
* **Claude Model:** `{CLAUDE_MODEL}`
* **Effort:** `{EFFORT}`
* **Thinking:** `{THINKING}`
* **Max output tokens:** `{MAX_OUTPUT_TOKENS}`
* **Service tier:** `{SERVICE_TIER}`
* **Strict JSON schema:** `{USE_STRICT_SCHEMA}` ({len(MODEL_COLUMNS)} required fields)
* **Prompt caching:** `{USE_PROMPT_CACHE}`
* **Replications per file:** `{REPLICATIONS}`
* **Retries on rate limit / overload:** `{NumberOfRetries}` (waiting `{WaitTime}` s between attempts), on top of the SDK's own two retries

_Current Claude models do not accept `temperature`, `top_p` or `top_k`; `EFFORT` is the equivalent control._
"""))

# Step 3: Process source files and Generate Report (JSON version)

This cell loops through all pdf, txt and md files in the SubFolder folder and calls the selected Claude model with the specified prompt for each one of them. With USE_STRICT_SCHEMA on, the API enforces the JSON structure, so the model cannot return anything but valid JSON with the expected fields. The results are then written in the output Excel file. This may take a long time depending on the length & complexity of the prompt and the number and size of your files.

If the file is successfully processed its name is written in the processed files Excel so it is not processed for the second time if the cell is run again.

Everything that happens is written to the log file, with a timestamp at the beginning of each entry: the prompt (logged once at the start of the batch), which file is being processed, every call to the LLM and its result (success or error), and every error. The raw answers returned by the API are written to the raw output file.

If a call fails with a rate limit (429) or overloaded (529) error it is retried NumberOfRetries times, waiting WaitTime seconds before each retry. The Anthropic SDK also retries these errors twice on its own, quietly, before the notebook ever sees them.

The controls set in Step 2 (effort, thinking, output ceiling, service tier, strict schema, prompt caching) are all sent with every call. Not every model accepts every control: if the model rejects one, that single setting is dropped, a note is written to the log, and the call is retried with the rest still in place. Once a setting has been dropped it stays dropped for the remainder of the run.

The token counts for each call - input, cached, thinking and output - are written to the log file and to the output Excel file, and a total for the whole batch is printed at the end. This is the data you need to work out what a run cost.

If REPLICATIONS is greater than 1, each file is sent that many times and each answer is written as its own row, numbered in the Run column. Comparing the rows for one file across runs is how you measure the stability of the coding.

Some files occasionally aren't successfully processed for some reason. In these cases it is worth running this cell again to see if they will be processed in the second run.

In [ ]:
# Step 3: Process source files and Generate Report (JSON version)
# ▶️ Run this cell to start the extraction process.
# This may take a long time depending on the number and size of your files.

from pypdf import PdfReader
import json

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# SUPPORTED INPUT FILE TYPES
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
SUPPORTED_EXTENSIONS = ('.pdf', '.txt', '.md')

# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPERS: LOGGING (every entry starts with a timestamp)
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def timestamp_now():
    """Returns the current time as a string used at the start of each log entry."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def log(message, to_screen=True):
    """Writes a timestamped entry to the log file and (by default) to the screen."""
    line = f"[{timestamp_now()}] {message}"
    try:
        with open(LOG_FILE_PATH, 'a', encoding='utf-8') as f:
            f.write(line + "\n")
    except Exception as e:
        print(f"⚠️ WARNING: Could not write to the log file: {e}")
    if to_screen:
        print(message)


def log_raw(filename, raw_text):
    """Writes the raw output received from an API call to the raw output file."""
    try:
        with open(RAW_OUTPUT_PATH, 'a', encoding='utf-8') as f:
            f.write("\n" + "=" * 80 + "\n")
            f.write(f"[{timestamp_now()}] RAW OUTPUT | file: {filename} | model: {CLAUDE_MODEL} | "
                f"effort: {EFFORT} | thinking: {THINKING}\n")
            f.write("=" * 80 + "\n")
            f.write((raw_text if raw_text is not None else "<empty response>") + "\n")
    except Exception as e:
        log(f"   ⚠️ WARNING: Could not write to the raw output file: {e}")


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPER: EXTRACT TEXT FROM A PDF
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def extract_text_from_pdf(path, max_pages=None):
    """
    Extracts text from a PDF using pypdf.
    max_pages: optional limit on the number of pages to read.
    """
    reader = PdfReader(path)
    text_chunks = []
    for i, page in enumerate(reader.pages):
        if max_pages is not None and i >= max_pages:
            break
        page_text = page.extract_text() or ""
        text_chunks.append(page_text)
    return "\n\n".join(text_chunks)


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPER: EXTRACT TEXT FROM A PLAIN TEXT OR MARKDOWN FILE
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def extract_text_from_textfile(path):
    """Reads a .txt or .md file whole, as UTF-8 (undecodable bytes are replaced)."""
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        return f.read()


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPER: EXTRACT TEXT FROM ANY SUPPORTED FILE
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def extract_text_from_file(path):
    """Dispatches to the right extractor based on the file extension."""
    extension = os.path.splitext(path)[1].lower()
    if extension == '.pdf':
        return extract_text_from_pdf(path)
    if extension in ('.txt', '.md'):
        return extract_text_from_textfile(path)
    raise ValueError(f"Unsupported file type: '{extension}'")


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPER: PARSE JSON FROM LLM RESPONSE
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def parse_llm_json(response_text, filename):
    """
    Parses JSON from an LLM response.

    Supports:
      - a single JSON object
      - a JSON array of objects
      - a JSON object wrapping the array under "results" (the JSON mode format)

    Returns a list of dicts.
    """
    # First try a straightforward JSON parse
    try:
        data = json.loads(response_text)
    except json.JSONDecodeError:
        # Fallback: try to extract the first JSON-like array/object with a regex
        # This handles cases where the model adds extra text accidentally.
        import re
        match = re.search(r'(\[.*\]|\{.*\})', response_text, re.DOTALL)
        if not match:
            log(f"   ⚠️ WARNING: Could not find valid JSON in response for {filename}.")
            return []
        try:
            data = json.loads(match.group(1))
        except json.JSONDecodeError:
            log(f"   ⚠️ WARNING: JSON still invalid after extraction for {filename}.")
            return []

    # Normalize to list[dict]
    if isinstance(data, dict):
        # JSON mode returns an object, so unwrap the array if it is nested
        for wrapper_key in ("results", "records", "items", "data", "papers"):
            if isinstance(data.get(wrapper_key), list):
                return [item for item in data[wrapper_key] if isinstance(item, dict)]
        return [data]
    if isinstance(data, list):
        # Keep only dict items
        return [item for item in data if isinstance(item, dict)]

    log(f"   ⚠️ WARNING: JSON root is not an object or array of objects for {filename}.")
    return []


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPER: DETECT RATE LIMIT AND OVERLOADED ERRORS (both worth retrying)
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def is_rate_limit_error(exception):
    """
    Returns True if the exception is a rate limit (HTTP 429) or an
    'overloaded' (HTTP 529) error. Both are transient and worth retrying.
    """
    retryable = tuple(
        cls for cls in (
            getattr(anthropic, "RateLimitError", None),    # 429
            getattr(anthropic, "OverloadedError", None),   # 529
        ) if cls is not None
    )
    if retryable and isinstance(exception, retryable):
        return True

    status = getattr(exception, 'status_code', None)
    if status is None:
        status = getattr(getattr(exception, 'response', None), 'status_code', None)
    if status in (429, 529):
        return True

    message = str(exception).lower()
    return ('rate limit' in message) or ('rate_limit' in message) or ('overloaded' in message)


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# HELPER: CALL THE CLAUDE API FORCING A JSON ANSWER
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# ------------------------------------------------------------
# BUILD THE JSON SCHEMA USED FOR STRUCTURED OUTPUTS
# ------------------------------------------------------------
# Structured Outputs guarantees that every record contains every field in
# MODEL_COLUMNS, so the repair logic in parse_llm_json() below rarely has to
# fire. The schema is derived from MODEL_COLUMNS automatically: change the
# coding manual and MODEL_COLUMNS in Step 2 and the schema follows.
def build_results_schema():
    """Turns MODEL_COLUMNS into the JSON schema the API expects."""
    properties = {
        column: {"type": SCHEMA_FIELD_TYPES.get(column, "string")}
        for column in MODEL_COLUMNS
    }
    record_schema = {
        "type": "object",
        "properties": properties,
        "required": list(MODEL_COLUMNS),   # strict mode: every field in every record
        "additionalProperties": False,
    }
    return {
        "type": "object",
        "properties": {"results": {"type": "array", "items": record_schema}},
        "required": ["results"],
        "additionalProperties": False,
    }


RESULTS_SCHEMA = build_results_schema()


# ------------------------------------------------------------
# ASSEMBLE THE API CALL FROM THE CONTROLS SET IN STEP 2
# ------------------------------------------------------------
# Not every model accepts every control. Haiku 4.5 does not accept adaptive
# thinking, older models do not accept every effort level, and so on. Rather
# than abandoning all the controls at the first complaint, the code below drops
# ONE offending parameter at a time and retries, so a single unsupported
# setting does not cost you the others. A dropped parameter stays dropped for
# the rest of the run.
DISABLED_PARAMS = set()


def build_request_kwargs(coding_manual, paper_text):
    """Builds the keyword arguments for client.messages.stream()."""
    # The coding manual is the same for every file, so it goes in its own
    # content block that can be marked for caching. The paper text follows it
    # in a second block, so the cached prefix is identical from call to call.
    manual_block = {"type": "text", "text": coding_manual}
    if "cache_control" not in DISABLED_PARAMS and USE_PROMPT_CACHE:
        manual_block["cache_control"] = {"type": "ephemeral"}

    kwargs = {
        "model": CLAUDE_MODEL,
        "max_tokens": MAX_OUTPUT_TOKENS,   # required by the API
        "system": CLAUDE_SYSTEM_PROMPT,
        "messages": [{
            "role": "user",
            "content": [manual_block, {"type": "text", "text": paper_text}],
        }],
    }

    # Effort - the main quality / cost / speed dial - and the strict schema
    # both live under output_config.
    output_config = {}
    if "effort" not in DISABLED_PARAMS and EFFORT:
        output_config["effort"] = EFFORT
    if ("json_schema" not in DISABLED_PARAMS and USE_STRICT_SCHEMA):
        output_config["format"] = {"type": "json_schema", "schema": RESULTS_SCHEMA}
    if output_config:
        kwargs["output_config"] = output_config

    # Thinking on (adaptive) or off (disabled)
    if "thinking" not in DISABLED_PARAMS and THINKING:
        kwargs["thinking"] = {"type": THINKING}

    # Priority capacity or not
    if ("service_tier" not in DISABLED_PARAMS
            and SERVICE_TIER and SERVICE_TIER != "auto"):
        kwargs["service_tier"] = SERVICE_TIER

    return kwargs


# Words that identify each parameter in an API error message
PARAM_ERROR_HINTS = {
    "effort": ("effort",),
    "thinking": ("thinking", "budget_tokens"),
    "json_schema": ("json_schema", "output_config.format", "structured output"),
    "cache_control": ("cache_control", "cache"),
    "service_tier": ("service_tier",),
}

# The order in which controls are given up when the error message does not say
# which one is at fault. Least important first. max_tokens is never dropped -
# the API requires it.
FALLBACK_DROP_ORDER = ["service_tier", "cache_control", "thinking", "json_schema", "effort"]


def looks_like_unsupported_parameter(exception):
    """
    True if the error means 'this model does not accept that setting'.
    Only 400-type errors qualify; a 400 for a different reason (for example a
    paper that is too long for the context window) has none of these phrases
    and is raised as normal.
    """
    if isinstance(exception, TypeError):
        return True
    try:
        if not isinstance(exception, anthropic.BadRequestError):
            return False
    except Exception:
        pass
    message = str(exception).lower()
    return any(phrase in message for phrase in (
        "not supported", "unsupported", "does not support", "unexpected keyword",
        "extra inputs are not permitted", "unknown field", "unrecognized",
        "not a valid", "cannot be disabled", "is not available",
    ))


def identify_offending_param(exception):
    """Guesses which control the API objected to, or None if it does not say."""
    message = str(exception).lower()
    for param, hints in PARAM_ERROR_HINTS.items():
        if param in DISABLED_PARAMS:
            continue
        if any(hint in message for hint in hints):
            return param
    return None


def next_param_to_drop():
    """The next control to give up when the error message is not specific."""
    for param in FALLBACK_DROP_ORDER:
        if param not in DISABLED_PARAMS:
            return param
    return None


def create_response(coding_manual, paper_text):
    """
    Calls the Messages API with the controls set in Step 2.
    If the model rejects one of them it is dropped and the call is retried,
    so one unsupported setting does not disable the rest.
    """
    while True:
        try:
            # Streaming is used because the SDK refuses non-streaming calls that
            # could run longer than 10 minutes (large MAX_OUTPUT_TOKENS). The
            # chunks are collected silently and the complete message is returned,
            # so the rest of the code sees exactly what messages.create() gave.
            with client.messages.stream(
                    **build_request_kwargs(coding_manual, paper_text)) as stream:
                return stream.get_final_message()
        except Exception as e:
            if not looks_like_unsupported_parameter(e):
                raise
            param = identify_offending_param(e) or next_param_to_drop()
            if param is None:
                raise   # nothing left to drop - the error is something else
            DISABLED_PARAMS.add(param)
            log(f"   ⚠️ '{param}' is not accepted by {CLAUDE_MODEL} "
                f"({type(e).__name__}: {e}). Dropping it and retrying; "
                f"the other controls stay in place.")


# ------------------------------------------------------------
# HELPER: THE TEXT OF A RESPONSE
# ------------------------------------------------------------
def response_text_of(response):
    """
    Returns the text the model produced. A Claude response is a list of
    content blocks; thinking blocks (if any) come first and are skipped.
    """
    parts = []
    for block in getattr(response, "content", []) or []:
        if getattr(block, "type", None) == "text":
            parts.append(getattr(block, "text", "") or "")
    return "".join(parts)


# ------------------------------------------------------------
# HELPER: READ THE TOKEN COUNTS OFF A RESPONSE
# ------------------------------------------------------------
def extract_usage(response):
    """
    Pulls the token counts out of a response.
    Note: Claude's usage.input_tokens is only the UNCACHED part of the input.
    The full input is input_tokens + cache_read_input_tokens +
    cache_creation_input_tokens, and that total is what is reported here as
    'input_tokens' so the column means the same thing whatever the caching.
    """
    usage = getattr(response, "usage", None)
    if usage is None:
        return {}
    uncached = getattr(usage, "input_tokens", 0) or 0
    cache_read = getattr(usage, "cache_read_input_tokens", 0) or 0
    cache_write = getattr(usage, "cache_creation_input_tokens", 0) or 0
    output_details = getattr(usage, "output_tokens_details", None)
    return {
        "input_tokens": uncached + cache_read + cache_write,
        "cached_tokens": cache_read,
        "cache_write_tokens": cache_write,
        "output_tokens": getattr(usage, "output_tokens", None),
        "reasoning_tokens": getattr(output_details, "thinking_tokens", None) if output_details else None,
        "service_tier": getattr(usage, "service_tier", None),
    }


def call_llm(coding_manual, paper_text, filename):
    """
    Calls the LLM for one file, forcing a JSON answer.
    On a rate limit error the call is retried NumberOfRetries times,
    waiting WaitTime seconds before each retry.
    Returns (response_text, usage_dict). response_text is None if every
    attempt failed.
    """
    total_attempts = NumberOfRetries + 1

    for attempt in range(1, total_attempts + 1):
        log(f"   → LLM CALL | file: {filename} | model: {CLAUDE_MODEL} | "
            f"attempt {attempt}/{total_attempts}", to_screen=False)
        try:
            response = create_response(coding_manual, paper_text)
            response_text = response_text_of(response)  # string with the model output
            usage = extract_usage(response)

            # Log every raw output received from the API
            log_raw(filename, response_text)

            # A response can come back cut short rather than raising an error
            stop_reason = getattr(response, "stop_reason", None)
            if stop_reason == "max_tokens":
                log(f"   ⚠️ INCOMPLETE RESPONSE | file: {filename} | the answer hit "
                    f"MAX_OUTPUT_TOKENS ({MAX_OUTPUT_TOKENS}). Raise it in Step 2, or lower EFFORT.")
            elif stop_reason == "refusal":
                log(f"   ⚠️ REFUSAL | file: {filename} | the model declined to answer.")
            elif stop_reason == "model_context_window_exceeded":
                log(f"   ⚠️ TOO LONG | file: {filename} | the paper does not fit in the context window.")

            log(f"   ✅ LLM CALL SUCCESS | file: {filename} | attempt {attempt}/{total_attempts} | "
                f"{len(response_text or '')} characters returned | "
                f"tokens in: {usage.get('input_tokens')} "
                f"(cached: {usage.get('cached_tokens')}) | "
                f"thinking: {usage.get('reasoning_tokens')} | "
                f"out: {usage.get('output_tokens')}", to_screen=False)
            return response_text, usage

        except Exception as e:
            if is_rate_limit_error(e) and attempt < total_attempts:
                log(f"   ⚠️ LLM CALL RATE LIMIT | file: {filename} | attempt {attempt}/{total_attempts} | "
                    f"{type(e).__name__}: {e} | retrying in {WaitTime} s")
                time.sleep(WaitTime)
                continue

            kind = "RATE LIMIT" if is_rate_limit_error(e) else "ERROR"
            log(f"   ❌ LLM CALL {kind} | file: {filename} | attempt {attempt}/{total_attempts} | "
                f"{type(e).__name__}: {e}")
            return None, {}

    return None, {}


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# MAIN PROCESSING LOOP
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
def process_files():
    """Main function to loop through the source files, call Claude, and save results."""

    # Check if the source folder exists
    if not os.path.isdir(PDF_FOLDER_PATH):
        log(f"❌ ERROR: The folder '{PDF_FOLDER_PATH}' was not found in your Google Drive.")
        return

    # Load or create the processed files DataFrame
    processed_files_df = pd.DataFrame(columns=['Filename'])
    if os.path.exists(PROCESSED_FILES_PATH):
        try:
            processed_files_df = pd.read_excel(PROCESSED_FILES_PATH)
            log(f"✅ Loaded processed files list from: {PROCESSED_FILES_PATH}")
        except Exception as e:
            log(f"❌ Error loading processed files list: {e}. Starting with an empty list.")

    # The files this notebook writes into the same folder must never be treated
    # as source files (the log and the raw output file are .txt).
    own_output_files = {
        os.path.basename(EXCEL_FILE_PATH),
        os.path.basename(PROCESSED_FILES_PATH),
        os.path.basename(LOG_FILE_PATH),
        os.path.basename(RAW_OUTPUT_PATH),
    }

    # List all supported source files (pdf, txt, md) in the specified folder
    source_files = [
        f for f in os.listdir(PDF_FOLDER_PATH)
        if f.lower().endswith(SUPPORTED_EXTENSIONS)
        and not f.startswith('~$')          # Excel/Word lock files
        and f not in own_output_files       # this notebook's own output files
    ]

    if not source_files:
        log(f"❌ No pdf, txt or md files found in '{PDF_FOLDER_PATH}'.")
        return

    # Filter out already processed files
    unprocessed_files = [f for f in source_files if f not in processed_files_df['Filename'].values]

    if not unprocessed_files:
        log("\n✅ All source files in the folder have already been processed.")
        return

    num_files = len(unprocessed_files)

    # --- Log the start of the batch and the prompt, once ---
    log("=" * 80, to_screen=False)
    log(f"BATCH START | Project: {ProjectName} | SubFolder: {SubFolder} | Model: {CLAUDE_MODEL} | "
        f"{len(source_files)} file(s) found, {num_files} to process", to_screen=False)
    log(f"CONTROLS | effort: {EFFORT} | thinking: {THINKING} | "
        f"max output tokens: {MAX_OUTPUT_TOKENS} | service tier: {SERVICE_TIER} | "
        f"strict schema: {USE_STRICT_SCHEMA} | prompt cache: {USE_PROMPT_CACHE} | "
        f"replications: {REPLICATIONS}", to_screen=False)
    log(f"SYSTEM PROMPT USED FOR THIS BATCH:\n{CLAUDE_SYSTEM_PROMPT}", to_screen=False)
    log(f"PROMPT USED FOR THIS BATCH:\n{CLAUDE_PROMPT}", to_screen=False)
    log("=" * 80, to_screen=False)

    print(f"Found {len(source_files)} source file(s). {num_files} unprocessed file(s) to process. Starting...")
    if REPLICATIONS > 1:
        print(f"Each file will be sent {REPLICATIONS} times (REPLICATIONS = {REPLICATIONS}).")
    print("-" * 50)

    i = 1

    # Running totals for the whole batch, so the run's cost can be worked out
    batch_usage = {"input_tokens": 0, "cached_tokens": 0, "cache_write_tokens": 0,
                   "reasoning_tokens": 0, "output_tokens": 0}

    # Loop through each unprocessed file
    for filename in unprocessed_files:
        full_path = os.path.join(PDF_FOLDER_PATH, filename)

        # Log which file is being processed
        log(f"🔄 PROCESSING {i}/{num_files} | file: {filename}")

        try:
            # --- Extract text from the source file locally ---
            file_text = extract_text_from_file(full_path)
            if not file_text.strip():
                log(f"   ⚠️ WARNING: No text could be extracted from {filename}. Skipping.")
                continue

            log(f"   ...extracted {len(file_text)} characters from {filename}.", to_screen=False)

            # The coding manual (cached) and the paper text (different each time)
            # are sent as two separate blocks - see build_request_kwargs().
            paper_text = f"---\nHere is the content from file: {filename}\n\n{file_text}"

            # --- Call the API, once per replication ---
            file_succeeded = False

            for run_number in range(1, REPLICATIONS + 1):
                if REPLICATIONS > 1:
                    log(f"   ↻ RUN {run_number}/{REPLICATIONS} | file: {filename}")

                response_text, usage = call_llm(CLAUDE_PROMPT, paper_text, filename)

                if response_text is None:
                    log(f"   ❌ FAILED: No response received for {filename} "
                        f"(run {run_number}/{REPLICATIONS}).")
                    continue

                # Keep a running total of the tokens used by this batch
                for key in batch_usage:
                    value = usage.get(key)
                    if isinstance(value, int):
                        batch_usage[key] += value

                print(f"   ...AI analysis complete.")

                # Parse JSON from Claude's response
                extracted_data = parse_llm_json(response_text, filename)

                if not extracted_data:
                    log(f"   ⚠️ WARNING: No structured JSON data could be extracted for "
                        f"{filename} (run {run_number}/{REPLICATIONS}).")
                    continue

                # Convert the extracted data into a pandas DataFrame
                new_df = pd.DataFrame(extracted_data)

                # Ensure all required columns are present, fill missing ones with blanks
                for col in EXCEL_COLUMNS:
                    if col not in new_df.columns:
                        new_df[col] = ''

                # Add the provenance and cost columns to the extracted data
                new_df['Source file'] = filename
                new_df['SubFolder'] = SubFolder
                new_df['Model'] = CLAUDE_MODEL
                new_df['Timestamp'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                new_df['Run'] = run_number
                new_df['Effort'] = ('not supported' if 'effort' in DISABLED_PARAMS else EFFORT)
                new_df['Thinking'] = ('not supported' if 'thinking' in DISABLED_PARAMS else THINKING)
                new_df['Input tokens'] = usage.get('input_tokens')
                new_df['Cached tokens'] = usage.get('cached_tokens')
                new_df['Thinking tokens'] = usage.get('reasoning_tokens')
                new_df['Output tokens'] = usage.get('output_tokens')

                # Reorder columns to match the Excel file and keep only those columns
                new_df = new_df[EXCEL_COLUMNS]

                # Append the results to the main Excel file
                master_df = pd.read_excel(EXCEL_FILE_PATH)
                updated_df = pd.concat([master_df, new_df], ignore_index=True)
                updated_df.to_excel(EXCEL_FILE_PATH, index=False)

                log(f"   ✅ SUCCESS: Appended {len(new_df)} row(s) for '{filename}' "
                    f"(run {run_number}/{REPLICATIONS}) to "
                    f"'{os.path.basename(EXCEL_FILE_PATH)}'.")
                file_succeeded = True

            # Only mark the file as done if at least one run produced usable data,
            # so a failed file is tried again the next time this cell is run.
            if not file_succeeded:
                log(f"   ❌ FAILED: no usable result for '{filename}'. Leaving it unprocessed.")
                continue

            # Add the processed filename to the tracking sheet
            new_processed_row = pd.DataFrame({'Filename': [filename]})
            processed_files_df = pd.concat([processed_files_df, new_processed_row], ignore_index=True)
            processed_files_df.to_excel(PROCESSED_FILES_PATH, index=False)
            log(f"   ...Marked '{filename}' as processed.")

        except Exception as e:
            log(f"   ❌ ERROR processing {filename}: {type(e).__name__}: {e}")

        finally:
            i = i + 1
            print("-" * 50)

    # --- Report what the batch used ---
    dropped = ", ".join(sorted(DISABLED_PARAMS)) if DISABLED_PARAMS else "none"
    total_tokens = batch_usage["input_tokens"] + batch_usage["output_tokens"]
    log(f"BATCH USAGE | input: {batch_usage['input_tokens']:,} "
        f"(of which read from cache: {batch_usage['cached_tokens']:,}, "
        f"written to cache: {batch_usage['cache_write_tokens']:,}) | "
        f"thinking: {batch_usage['reasoning_tokens']:,} | "
        f"output: {batch_usage['output_tokens']:,} | "
        f"total billable: {total_tokens:,} | controls dropped: {dropped}", to_screen=False)
    log(f"BATCH END | Project: {ProjectName} | SubFolder: {SubFolder}", to_screen=False)

    print("\n🎉 All unprocessed source files have been processed.")
    print(f"🔢 Tokens used - input: {batch_usage['input_tokens']:,} "
          f"(read from cache: {batch_usage['cached_tokens']:,}, "
          f"written to cache: {batch_usage['cache_write_tokens']:,}), "
          f"thinking: {batch_usage['reasoning_tokens']:,}, "
          f"output: {batch_usage['output_tokens']:,}")
    if DISABLED_PARAMS:
        print(f"⚠️ Controls not accepted by {CLAUDE_MODEL} and therefore dropped: {dropped}")
    print(f"📝 Log file: {LOG_FILE_PATH}")
    print(f"📄 Raw output file: {RAW_OUTPUT_PATH}")


# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---
# EXECUTE THE MAIN FUNCTION
# --- --- --- --- --- --- --- --- --- --- --- --- --- --- ---

process_files()

# Step 4 (Optional): View Final Report

This cell shows the contents of output Excel file.

In [ ]:
# Step 4: View Final Report
# Run this cell to display the contents of the final Excel file.

import pandas as pd
from IPython.display import display

try:
    final_df = pd.read_excel(EXCEL_FILE_PATH)
    print(f"Displaying the contents of '{os.path.basename(EXCEL_FILE_PATH)}':")
    display(final_df)

    # A short summary of what the run cost, if the token columns are present
    token_columns = ["Input tokens", "Cached tokens", "Thinking tokens", "Output tokens"]
    if all(col in final_df.columns for col in token_columns):
        totals = final_df[token_columns].fillna(0).sum().astype(int)
        print("\nTokens recorded in this file:")
        for name, value in totals.items():
            print(f"  {name}: {value:,}")

    # If the file was coded more than once, show how many runs each file has
    if "Run" in final_df.columns and final_df["Run"].nunique(dropna=True) > 1:
        print("\nRuns per source file (for checking stability across replications):")
        display(final_df.groupby("Source file")["Run"].nunique().rename("Number of runs"))
except FileNotFoundError:
    print(f"❌ The file '{EXCEL_FILE_PATH}' could not be found.")
except Exception as e:
    print(f"An error occurred while reading the Excel file: {e}")